## House Robber II (circular street)

### Problem

You are a robber on a street where the houses are arranged in a **circle**.
Each house holds a non-negative amount of money. Adjacent houses have linked
alarms, so you cannot rob two neighboring houses. Because the street is a
circle, the **first and last houses are neighbors** too.

Given `nums`, where `nums[i]` is the money in house `i`, return the maximum
amount you can rob without robbing two adjacent houses.

### Function signature

    def rob(nums: list[int]) -> int:

### Constraints

- $1 \le n \le 100$, where $n$ is `len(nums)`
- $0 \le$ `nums[i]` $\le 1000$

### Examples

| `nums`         | Output | Why                                                             |
| -------------- | ------ | --------------------------------------------------------------- |
| `[2, 3, 2]`    | `3`    | Houses 0 and 2 are neighbors in the circle, so rob only house 1 |
| `[1, 2, 3, 1]` | `4`    | Rob houses 0 and 2: $1 + 3 = 4$                                 |
| `[1, 2, 3]`    | `3`    | Rob house 2 alone                                               |
| `[5]`          | `5`    | A single house has no neighbors                                 |

### Invariant (fill this in before coding)

- State: ...
- Update rule: ...
- What makes the circle different from the straight street: ...


In [ ]:
def rob(nums: list[int]) -> int:
    if len(nums) == 1:
        return nums[0]

    def rob_helper(lo, hi):
        picked, skipped = 0, 0
        # Invariant - At the beginning of each iteration -
        # picked = the maximum loot possible for the houses starting from lo, if the previous house was picked,
        # skipped = the maximumum loot possible for the houses starting lo, if the previous house was skipped.
        for house in nums[lo:hi]:
            picked, skipped = skipped + house, max(picked, skipped)
        return max(picked, skipped)

    return max(rob_helper(1, len(nums)), rob_helper(0, len(nums) - 1))

In [ ]:
assert rob([2, 3, 2]) == 3
assert rob([1, 2, 3, 1]) == 4
assert rob([1, 2, 3]) == 3
assert rob([5]) == 5

Claude's feedback-

Your code isn't bad. It's correct, and the approach of running the straight-street solution twice is exactly the right idea. I checked the edge cases: one house, two houses, and all zeros all work. What needs work is the explanation inside it, not the logic. Here's the strict review.

**1. The invariant is imprecise.** "Starting from lo" says where the houses begin, but not where they end, so it doesn't say which houses the totals cover. A sharper version: at the start of the iteration for house $i$, `picked` is the best total over houses `lo` to $i-1$ given that house $i-1$ was robbed, and `skipped` is the best total over the same houses given that house $i-1$ was not robbed. Also notice your starting values: before the first house there is no previous house, so `picked = 0` means "nothing robbed yet". That's safe only because `picked` is never added to a house; it only goes into `max`.

**2. The correctness argument is missing, and it's the most important line.** Why does it work to take the best of two ranges? In any valid robbery of the circle, house 0 and house $n-1$ are neighbors, so at least one of them is not robbed. That means every valid robbery is also a valid robbery of either the street without house 0, or the street without house $n-1$. So the best of those two straight streets is the answer. That reasoning should be a comment, because anyone reading the code needs it.

**3. The `len(nums) == 1` check is a legitimate base case, but say why.** Without it, one house gives 0: both ranges drop the only house. The "drop one end" argument assumes house 0 and house $n-1$ are different houses, and with one house they're the same house. That's a real structural edge case, not a messy flag, so one comment line is all it needs.

**4. Python details.**

- `nums[lo:hi]` copies that part of the list into a new list, so each call uses $O(n)$ extra memory. For 100 houses it doesn't matter, but it's the slicing gotcha worth knowing. `itertools.islice` or a loop over indices avoids the copy.
- `rob_helper` doesn't say what it does. A name like `rob_street` does. Better still, have it take the list of houses, so it becomes exactly your House Robber I solution, reused.
- Small fixes: a space after the comma in the `max` call, the "maximumum" typo, and type hints on the helper.

Here's a version with those fixes:

```python
from itertools import islice


def rob(nums: list[int]) -> int:
    """Max loot from houses in a circle, never robbing two neighbors."""
    n = len(nums)
    # One house: it is both first and last, so "drop one end" would drop it.
    if n == 1:
        return nums[0]

    def rob_street(lo: int, hi: int) -> int:
        """House Robber I on the straight street nums[lo:hi], without copying."""
        picked, skipped = 0, 0
        # Invariant, at the start of the iteration for house i:
        #   picked  = best total over houses lo..i-1, with house i-1 robbed
        #   skipped = best total over houses lo..i-1, with house i-1 not robbed
        for house in islice(nums, lo, hi):
            picked, skipped = skipped + house, max(picked, skipped)
        return max(picked, skipped)

    # House 0 and house n-1 are neighbors, so every valid robbery leaves at
    # least one of them unrobbed: the answer is the better of the two streets.
    return max(rob_street(1, n), rob_street(0, n - 1))
```

One test I'd add is your example from earlier, 2, 1, 1, 9, now as a circle. On the straight street the answer was 11, robbing the first and last houses. In a circle those two are neighbors, so the answer drops to 10, which is a nice check that the circle rule actually matters:

```python
assert rob([2, 1, 1, 9]) == 10   # straight street gives 11; the circle forbids 2 + 9
assert rob([2, 3, 2]) == 3
assert rob([1, 2, 3, 1]) == 4
assert rob([5]) == 5
assert rob([0, 0, 0]) == 0
```

**Grade:** full marks for correctness and approach, with points off for the vague invariant and the missing correctness argument. Next, can you explain in one sentence why `islice(nums, lo, hi)` doesn't copy the list but `nums[lo:hi]` does?


In [ ]:
from itertools import islice


def rob(nums: list[int]) -> int:
    """Max loot from houses in a circle, never robbing two neighbors."""
    n = len(nums)
    # One house: it is both first and last, so "drop one end" would drop it.
    if n == 1:
        return nums[0]

    def rob_street(lo: int, hi: int) -> int:
        """House Robber I on the straight street nums[lo:hi], without copying."""
        picked, skipped = 0, 0
        # Invariant, at the start of the iteration for house i:
        #   picked  = best total over houses lo..i-1, with house i-1 robbed
        #   skipped = best total over houses lo..i-1, with house i-1 not robbed
        for house in islice(nums, lo, hi):
            picked, skipped = skipped + house, max(picked, skipped)
        return max(picked, skipped)

    # House 0 and house n-1 are neighbors, so every valid robbery leaves at
    # least one of them unrobbed: the answer is the better of the two streets.
    return max(rob_street(1, n), rob_street(0, n - 1))

In [ ]:
assert rob([2, 1, 1, 9]) == 10  # straight street gives 11; the circle forbids 2 + 9
assert rob([2, 3, 2]) == 3
assert rob([1, 2, 3, 1]) == 4
assert rob([5]) == 5
assert rob([0, 0, 0]) == 0